# Jour 3 : du texte OCR aux événements du calendrier

**Objectif de la journée** : transformer la sortie de l'OCR (le fichier `ocr_<alias>.json` produit au jour 2)
en **événements** au format du cours, puis les **publier** dans l'agenda partagé de votre squad.

```
ocr_champion.json  →  parsing (VOTRE travail)  →  événements  →  CalDAV (fourni)
```

Un événement suit le schéma du cours :

```python
{"title": "Équipe A", "date": "2026-03-12", "start_time": "14:00", "end_time": "18:00",
 "location": None, "confidence": 0.85}
```

(`start_time`, `end_time` et `location` peuvent valoir `None`.)

Champs **optionnels**, pour les plannings à codes (équipes, postes) : `"person"` (la personne concernée, lue
sur la ligne), `"all_day"` (`True` : journée entière, ex. un repos) et `"overnight"` (`True` : poste de nuit,
la fin tombe le lendemain).

**Règles de confiance** (chaque événement reçoit un score entre 0 et 1) :

| Confiance | Décision |
|---|---|
| ≥ 0,8 | **pré-coché** : proposé et accepté par défaut |
| 0,5 – 0,8 | **à vérifier** : proposé, mais l'utilisateur doit confirmer |
| < 0,5 | **non proposé** |

> Le **rappel** reste la métrique qui compte : un créneau raté est un créneau perdu. Mieux vaut proposer
> un événement douteux « à vérifier » que le laisser tomber en silence.

La publication CalDAV (`caldav_publier.py`) est **fournie** : vous écrivez le parsing.

## Organisation de la squad : une partie par membre

Le notebook est long pour une seule personne : **répartissez les fonctions**, chacun dans **sa copie** du
notebook (*Fichier → Enregistrer une copie*), puis le ou la lead colle les cellules terminées dans le notebook
commun. Un même `.ipynb` modifié à plusieurs (Colab partagé ou git) donne des conflits.

| Membre | Fonctions (section 3) | Tests de la section 4 |
|---|---|---|
| **Heures et texte** | `normaliser`, `extraire_heures`, `CODES` + `traduire_code` | normaliser, extraire_heures, traduire_code |
| **Dates** | `extraire_date`, `extraire_mois_annee` | extraire_date, bruit d'OCR |
| **Géométrie** | réglage `CLASSES_*`, `associer`, `associer_ligne` | associer, associer_ligne |
| **Lead : assemblage** | `confiance`, `parser`, seuils (section 6), publication (section 7) | confiance, puis tout le reste |

À trois : le lead prend aussi `CODES` et `traduire_code`.

Chaque fonction a ses exemples et son test : la section 4 affiche `A FAIRE` pour une fonction pas encore
écrite, sans bloquer les autres. Seule dépendance : `extraire_date` appelle `normaliser`. En attendant la
vraie version, le membre « Dates » remplace dans **sa copie** le `raise NotImplementedError` de `normaliser`
par `return texte`.

**Parcours conseillé**

1. **Avant midi** : heures, dates, `associer` dans le cas le plus simple de votre planning, `parser`, puis une
   publication en **simulation** (section 7). Objectif : au moins une photo de bout en bout.
2. **Après-midi** : plannings à codes (`traduire_code`, `associer_ligne`, date par la colonne), `confiance`
   affinée, cas difficiles relevés sur vos photos, publication réelle sans doublon.
3. **Bonus** : en-tête sur deux mois (section 3.3), API `POST /parse` (section 8).

## 1. Paramètres

In [32]:
OCR_JSON = "ocr_challenger_nathalie.json"       # fichier produit par test_ocr_champion.ipynb
CREDENTIALS = "identifiants-<login>.json"   # fichier téléchargé sur votre profil du portail
PUBLIER = False                      # True = vraie publication sur Radicale (après la simulation !)
ANNEE = 2026                         # année par défaut quand elle n'apparaît pas sur le planning

## 2. Chargement des résultats OCR

Le fichier a cette forme :

```json
{"modele": "flexistaff_team-rocket@champion", "version": "3", "classes": {"0": "..."},
 "images": [{"image": "p1.jpg", "largeur": 1200, "hauteur": 800,
             "zones": [{"classe": "event", "confiance_detection": 0.91,
                        "x1": 10, "y1": 20, "x2": 110, "y2": 60,
                        "texte": "14h-18h Équipe A", "confiance_ocr": 0.87, "lignes_ignorees": 0,
                        "lignes": [{"texte": "14h-18h Équipe A", "confiance": 0.87,
                                    "x1": 12, "y1": 24, "x2": 104, "y2": 40}]}]}]}
```

`lignes` : les lignes visuelles du texte de la zone, de haut en bas, avec leur position dans l'image
(absent d'un fichier produit avant le 18/11 : relancez alors `test_ocr_champion.ipynb`).

In [33]:
import json
from collections import Counter
from pathlib import Path

with open(OCR_JSON, encoding="utf-8") as f:
    ocr = json.load(f)

print("Modèle :", ocr["modele"], "| version", ocr["version"])
print("Classes :", ocr["classes"])
for img in ocr["images"]:
    compte = Counter(z["classe"] for z in img["zones"])
    print(f"{img['image']} ({img['largeur']}x{img['hauteur']}) : {dict(compte)}")

Modèle : flexistaff_team-magma@challenger_nathalie | version 11
Classes : {'0': 'Date', '1': 'Cellule', '2': 'Nom', '3': 'Titre', '4': 'Cellule_code'}
lot-test_p000_IMG_2908.jpg (1600x1200) : {'Cellule': 32, 'Nom': 9, 'Date': 7, 'Titre': 1}
lot-test_p001_IMG_2909.jpg (1600x1200) : {'Cellule': 39, 'Nom': 8, 'Date': 9, 'Titre': 1}
lot-test_p002_IMG_2910.jpg (1600x1200) : {'Cellule': 34, 'Titre': 1}
lot-test_p003_IMG_2911.jpg (1600x1200) : {'Cellule': 34, 'Titre': 1, 'Date': 2}
lot-test_p004_IMG_2912.jpg (1600x1200) : {'Cellule': 18, 'Nom': 9, 'Date': 3, 'Titre': 1}
lot-test_p005_IMG_2913.jpg (1600x1200) : {'Cellule': 33, 'Titre': 1, 'Date': 1}
lot-test_p006_IMG_2915.jpg (1600x1200) : {'Cellule': 30, 'Nom': 8, 'Date': 7, 'Titre': 1}
lot-test_p007_IMG_2916.jpg (1600x1200) : {'Cellule_code': 166, 'Nom': 12, 'Date': 15}
lot-test_p008_IMG_2917.jpg (1600x1200) : {'Cellule': 30, 'Titre': 1}
lot-test_p009_IMG_2918.jpg (1600x1200) : {'Cellule_code': 335, 'Nom': 11, 'Date': 31, 'Titre': 1}


### Un coup d'œil aux textes lus, classe par classe

**Vos classes sont les vôtres** (pas forcément `date_cell` / `event`…). Repérez ici laquelle porte la date,
laquelle porte l'heure, laquelle porte le titre, et adaptez les constantes de la section suivante.

In [ ]:
for img in ocr["images"][:2]:  # les 2 premières images suffisent pour se faire une idée
    print("=====", img["image"])
    for z in sorted(img["zones"], key=lambda z: (z["y1"], z["x1"])):
        print(f"{z['classe']:<16} ({z['x1']:>4},{z['y1']:>4})-({z['x2']:>4},{z['y2']:>4})  {z['texte']!r}")

## 3. Votre travail : les fonctions de parsing

Complétez les fonctions ci-dessous (chaque membre les siennes, voir « Organisation de la squad »), en vérifiant
à chaque fois leurs exemples.
Les exemples (`>>>`) montrent le résultat attendu ; la section 4 contient des tests automatiques.

Commencez par l'adaptation à vos classes : indiquez quelle classe joue quel rôle.

In [ ]:
# TODO : adaptez ces noms de classes à VOS labels (voir le résumé ci-dessus)
CLASSES_EVENEMENT = {"event"}       # zones qui décrivent un créneau (heures, titre)
CLASSES_DATE = {"date_cell"}        # zones qui portent un jour (numéro, ou date complète)
CLASSES_ENTETE_MOIS = {"month_header"}  # zones qui portent le mois (et parfois l'année)
CLASSES_ENTETE_JOUR = {"day_header_row"}  # en-têtes de colonnes (Lun, Mar...) ; peut ne pas exister chez vous
CLASSES_PERSONNE = set()  # optionnel : planning à codes, zones qui portent le nom de la personne (ex. {"agent"}).
                          # Si CLASS_MAP les envoie vers le rôle `resource_header`, c'est cette ligne-là.

### 3.1 `normaliser(texte)` : corriger les confusions de l'OCR

L'OCR confond souvent lettres et chiffres. On corrige **seulement dans un contexte numérique**
(un `O` au milieu de « Octobre » ne doit pas devenir `0`).

Règles de départ : `O` → `0`, `I` ou `l` → `1`, `Z` → `2`, et espaces parasites autour de `/`, `:`, `h`, `-`.

In [ ]:
import re


def normaliser(texte: str) -> str:
    """Corrige les confusions lettre/chiffre de l'OCR, uniquement en contexte numérique.

    >>> normaliser("1Z/O3")
    '12/03'
    >>> normaliser("I4h-18h")
    '14h-18h'
    >>> normaliser("Octobre")      # pas de chiffre autour : on ne touche à rien
    'Octobre'
    >>> normaliser("14 h 30")
    '14h30'
    """
    # TODO : ex. remplacer O/I/l/Z seulement s'ils sont collés à un chiffre ou à un séparateur de date/heure,
    # puis resserrer les espaces autour de « h », « : », « / » et « - ».
    # Provisoire, pour les autres membres en attendant la vraie version : remplacez la ligne ci-dessous
    # par `return texte` (le texte inchangé).
    raise NotImplementedError

### 3.2 `extraire_heures(texte)` : début et fin

Formats à couvrir : `14h-18h`, `14:00-18:00`, `de 14h à 18h`, `9h30`, `8h30 - 12h`…
Retour : `(start_time, end_time)` au format `"HH:MM"`, `None` si absent. Une heure seule donne `(début, None)`.

In [ ]:
def extraire_heures(texte: str):
    """Renvoie (start_time, end_time) au format "HH:MM" ; None pour ce qui est absent.

    >>> extraire_heures("14h-18h")
    ('14:00', '18:00')
    >>> extraire_heures("de 14h à 18h Réunion")
    ('14:00', '18:00')
    >>> extraire_heures("14:00-18:00")
    ('14:00', '18:00')
    >>> extraire_heures("9h30")
    ('09:30', None)
    >>> extraire_heures("Congé")
    (None, None)
    """
    # TODO : une expression régulière (re) qui capture 1 ou 2 heures, avec minutes optionnelles.
    # Pensez à valider : 0 <= heure <= 23 et 0 <= minutes <= 59 (un « 25h » est une erreur d'OCR).
    raise NotImplementedError

### 3.3 `extraire_date(texte, annee=ANNEE, mois=None)` : la date

Selon vos classes, la date peut venir :

- d'une **cellule de date** (`12`, `Mar 12`, `12/03`, `12/03/2026`) ;
- d'un **en-tête de colonne** (`Mar 12/03`) ;
- d'un **en-tête de mois** (`Mars 2026`) : il donne le mois (et l'année) quand la cellule n'a que le numéro du jour.

Retour : une date `"YYYY-MM-DD"` ou `None` si elle est introuvable ou impossible (31/02 → `None`).
Le paramètre `mois` (entier) sert quand le texte ne contient que le jour. **Adaptez cette fonction à votre planning.**

**Bonus : en-tête sur deux mois** (« Novembre 2027 / Décembre 2027 ») : les numéros de jour qui **repartent à 1**
après avoir atteint 28-31 appartiennent au mois **suivant** (et à l'année suivante après décembre).
Gardez donc le dernier numéro lu dans la ligne d'en-tête pour détecter le passage.

In [ ]:
MOIS = {"janvier": 1, "fevrier": 2, "février": 2, "mars": 3, "avril": 4, "mai": 5, "juin": 6, "juillet": 7,
        "aout": 8, "août": 8, "septembre": 9, "octobre": 10, "novembre": 11, "decembre": 12, "décembre": 12}


def extraire_date(texte: str, annee: int = ANNEE, mois: int | None = None):
    """Renvoie une date "YYYY-MM-DD", ou None.

    >>> extraire_date("Mar . 1Z/O3")
    '2026-03-12'
    >>> extraire_date("12/03/2025")
    '2025-03-12'
    >>> extraire_date("12", mois=3)           # numéro du jour seul + mois connu par l'en-tête
    '2026-03-12'
    >>> extraire_date("12 mars")
    '2026-03-12'
    >>> extraire_date("31/02") is None        # date impossible
    True
    """
    # TODO : appelez normaliser(), puis essayez les formats du plus précis au moins précis.
    # Validez avec datetime.date(annee, mois, jour) : une ValueError => None.
    raise NotImplementedError


def extraire_mois_annee(texte: str, annee: int = ANNEE):
    """Lit un en-tête de mois (« Mars 2026 ») et renvoie (annee, mois), ou (annee, None).

    >>> extraire_mois_annee("Mars 2026")
    (2026, 3)

    Bonus : un en-tête sur deux mois (« Novembre 2027 / Décembre 2027 ») donne deux couples : voyez comment
    les distinguer (les jours qui repartent à 1 basculent sur le mois suivant).
    """
    # TODO : cherchez un nom de mois (MOIS) et une année à 4 chiffres.
    raise NotImplementedError

### 3.4 `associer(zones)` : relier un événement à sa date

Une zone « événement » ne contient généralement pas sa date : il faut la **déduire de la géométrie**
(coordonnées `x1, y1, x2, y2`). Pistes, à choisir selon **votre** type de planning :

- **Calendrier mural** : la zone événement est *à l'intérieur* de la cellule de date (recouvrement des boîtes,
  ex. part de l'aire de l'événement contenue dans la cellule).
- **Planning en colonnes** : la date est l'en-tête de la **même colonne** (centres `x` alignés).
- **Planning en lignes** : la date est dans la cellule la plus proche **à gauche** sur la même ligne (centres `y` alignés).

Retour : pour chaque zone événement, la zone « date » associée (ou `None`), avec si possible un indicateur
de fiabilité (recouvrement faible = moins sûr).
- **Planning à codes** (équipe × jours) : le texte d'une cellule (`M06`, `N08`, `RH`...) ne contient **ni la
  personne ni la date**. La **personne** se trouve par la **ligne** (zone « agent » dont le centre vertical
  `y` est aligné avec celui de l'événement) ; la **date** par la **colonne** (en-tête de date dont le centre
  horizontal `x` est aligné), quand la cellule de date ne contient pas l'événement.

**Pistes selon les classes de votre squad** (vos boîtes ne suivent pas toutes le cas « l'événement est dans
sa cellule de date ») :

- **Team Rocket** : `Date_case` encadre le **texte** de la date, pas la case. En planning hebdo, c'est l'en-tête
  de la colonne ; en grille de mois, c'est le numéro du jour, en haut de la case. Dans les deux cas, la date
  d'un `Evenement` est la `Date_case` **de la même colonne** (recouvrement horizontal) la plus proche
  **au-dessus** de lui.
- **Team Magma** : en planning hebdo, `Date` est la case de la date (en-tête de colonne). En grille de mois, il
  n'y a pas de `Date` : la `Cellule` est la case du jour entière, et **son texte commence par le numéro du jour**
  (« 12 09:00 Réunion 14:00 Visite »). Lisez ce numéro, puis faites un événement par ligne : la zone a une
  liste `lignes` (une par ligne visuelle, de haut en bas, avec sa position), la première commence par le numéro
  (seul, ou suivi du premier événement si l'OCR les a lus sur la même ligne). Une `Cellule` qui ne contient que son numéro est un jour vide : pas d'événement.
- **Team Aqua** : `ref_semaine_planning` contient **toute la rangée** des jours (« LUNDI 30/12 MARDI 31/12… »),
  parfois avec le titre. Le JSON ne donne que le texte de la boîte, pas la position de chaque date : lisez
  les dates dans l'ordre, puis partagez la **largeur** de la boîte en autant de colonnes égales (la **hauteur**
  si la rangée est verticale, planning transposé). Une `horaire_lieu_jour_tache` prend la date de la colonne
  où tombe son centre. Une case sans texte est un jour vide : pas d'événement. En grille de mois, faites comme
  Magma (numéro du jour en tête du texte).

In [ ]:
def aire(z) -> float:
    return max(0, z["x2"] - z["x1"]) * max(0, z["y2"] - z["y1"])


def recouvrement(a, b) -> float:
    """Part de l'aire de `a` qui est contenue dans `b` (0 à 1).

    >>> a = {"x1": 10, "y1": 10, "x2": 20, "y2": 20}
    >>> b = {"x1": 0, "y1": 0, "x2": 100, "y2": 100}
    >>> recouvrement(a, b)
    1.0
    """
    ix = max(0, min(a["x2"], b["x2"]) - max(a["x1"], b["x1"]))
    iy = max(0, min(a["y2"], b["y2"]) - max(a["y1"], b["y1"]))
    return (ix * iy) / aire(a) if aire(a) else 0.0


def associer(zones: list[dict]):
    """Relie chaque zone événement à sa zone date, par la géométrie.

    Renvoie une liste de tuples (zone_evenement, zone_date_ou_None, score) où score est entre 0 et 1.

    >>> evt = {"classe": "event", "x1": 12, "y1": 30, "x2": 90, "y2": 50, "texte": "14h-18h A"}
    >>> jour = {"classe": "date_cell", "x1": 0, "y1": 0, "x2": 100, "y2": 100, "texte": "12"}
    >>> associer([evt, jour])[0][1]["texte"]
    '12'
    """
    # TODO : séparez les zones par rôle (CLASSES_EVENEMENT / CLASSES_DATE), puis pour chaque événement
    # choisissez la zone date qui convient (recouvrement maximal, ou colonne/ligne alignée).
    # TODO (après-midi, planning à codes) : si aucune cellule de date ne contient l'événement, prenez l'en-tête de date
    # de la même colonne (centres x alignés) : le score est plus faible qu'avec un vrai recouvrement.
    raise NotImplementedError

def associer_ligne(zone_evt: dict, zones: list[dict]):
    """Renvoie la zone en tête de la même ligne que l'événement (à gauche, centres verticaux alignés), ou None.

    Générique : `zones` peut être la liste des personnes (planning classique : le nom est au début de la ligne)
    ou celle des dates (planning transposé : la date est au début de la ligne).

    >>> evt = {"x1": 120, "y1": 40, "x2": 180, "y2": 60, "texte": "M06"}
    >>> p1 = {"classe": "agent", "x1": 0, "y1": 38, "x2": 100, "y2": 62, "texte": "Roux Inès"}
    >>> p2 = {"classe": "agent", "x1": 0, "y1": 64, "x2": 100, "y2": 88, "texte": "Le Gall Tom"}
    >>> associer_ligne(evt, [p1, p2])["texte"]
    'Roux Inès'
    >>> d1 = {"classe": "Date", "x1": 0, "y1": 38, "x2": 100, "y2": 62, "texte": "23 mars"}
    >>> associer_ligne(evt, [d1])["texte"]          # même fonction, avec des dates
    '23 mars'
    """
    # Le milieu vertical de l'événement : à mi-chemin entre son haut (y1) et son bas (y2)
    milieu_evt = (zone_evt["y1"] + zone_evt["y2"]) / 2

    # La zone d'en-tête couvre toute la hauteur de sa ligne : l'événement est sur cette ligne si son milieu
    # tombe entre le haut et le bas de la zone. Elle doit aussi être À GAUCHE de l'événement : c'est le début
    # de la ligne. Ce que contient la zone (un nom, une date) ne compte pas ici.
    for zone in zones:
        meme_ligne = zone["y1"] <= milieu_evt <= zone["y2"]
        a_gauche = zone["x2"] <= zone_evt["x1"] + 10  # 10 px de tolérance : les boîtes se touchent
        if meme_ligne and a_gauche:
            return zone

    return None  # aucune zone en tête de cette ligne : pas de réponse fiable


def associer_colonne(zone_evt: dict, zones: list[dict]):
    """Renvoie la zone en tête de la même colonne que l'événement (au-dessus, centres horizontaux alignés), ou None.

    Générique : `zones` peut être la liste des dates (planning classique : la date est en haut de la colonne)
    ou celle des personnes (planning transposé : le nom est en haut de la colonne).

    >>> evt = {"x1": 120, "y1": 40, "x2": 180, "y2": 60, "texte": "M06"}
    >>> d1 = {"classe": "Date", "x1": 110, "y1": 0, "x2": 190, "y2": 30, "texte": "23/03"}
    >>> d2 = {"classe": "Date", "x1": 190, "y1": 0, "x2": 270, "y2": 30, "texte": "24/03"}
    >>> associer_colonne(evt, [d1, d2])["texte"]
    '23/03'
    >>> p1 = {"classe": "Nom", "x1": 110, "y1": 0, "x2": 190, "y2": 30, "texte": "Roux Inès"}
    >>> associer_colonne(evt, [p1])["texte"]        # même fonction, avec des personnes
    'Roux Inès'
    """
    # Le milieu horizontal de l'événement : à mi-chemin entre son bord gauche (x1) et son bord droit (x2)
    milieu_evt = (zone_evt["x1"] + zone_evt["x2"]) / 2

    # La zone d'en-tête couvre toute la largeur de sa colonne : l'événement est dans cette colonne si son milieu
    # tombe entre le bord gauche et le bord droit de la zone. Elle doit aussi être AU-DESSUS de l'événement :
    # c'est le haut de la colonne. Ce que contient la zone (un nom, une date) ne compte pas ici.
    for zone in zones:
        meme_colonne = zone["x1"] <= milieu_evt <= zone["x2"]
        au_dessus = zone["y2"] <= zone_evt["y1"] + 10  # 10 px de tolérance : les boîtes se touchent
        if meme_colonne and au_dessus:
            return zone

    return None  # aucune zone en tête de cette colonne : pas de réponse fiable


def decouper_cellule(zone: dict):
    """Découpe une Cellule de Magma en (numéro du jour, liste des événements de la case), ou (None, [zone]).

    - Grille de mois : la 1re ligne commence par le numéro du jour, puis chaque ligne est un événement.
    - Planning hebdo : pas de numéro ; la case entière est UN événement, renvoyé tel quel.
    Comme associer_ligne, on renvoie les lignes entières (texte, confiance, position), pas seulement le texte :
    parser en a besoin, par exemple pour la confiance.

    >>> case = {"texte": "27 9h15 Atelier", "lignes": [{"texte": "27", "confiance": 0.99},
    ...                                                 {"texte": "9h15 Atelier", "confiance": 0.78}]}
    >>> numero, evenements = decouper_cellule(case)
    >>> numero, [e["texte"] for e in evenements]
    (27, ['9h15 Atelier'])
    >>> evenements[0]["confiance"]                  # la ligne entière, avec sa confiance
    0.78
    >>> case = {"texte": "16 10:15 Coiffeur 18:00 Cinéma", "lignes": [{"texte": "16 10:15 Coiffeur", "confiance": 0.9},
    ...                                                                {"texte": "18:00 Cinéma", "confiance": 0.8}]}
    >>> numero, evenements = decouper_cellule(case)
    >>> numero, [e["texte"] for e in evenements]   # le 1er événement était sur la ligne du numéro
    (16, ['10:15 Coiffeur', '18:00 Cinéma'])
    >>> decouper_cellule({"texte": "23", "lignes": [{"texte": "23", "confiance": 1.0}]})     # jour vide
    (23, [])
    >>> hebdo = {"texte": "9h à 18h Hangar C", "lignes": [{"texte": "9h à 18h"}, {"texte": "Hangar C"}]}
    >>> decouper_cellule(hebdo)[0], decouper_cellule(hebdo)[1][0]["texte"]   # la case entière, telle quelle
    (None, '9h à 18h Hangar C')
    >>> decouper_cellule({"texte": "", "lignes": []})                         # l'OCR n'a rien lu
    (None, [])
    """
    # Une case où l'OCR n'a rien lu : aucun événement (et surtout pas un événement vide)
    if zone["texte"].strip() == "":
        return None, []

    # Les lignes de la case, de haut en bas. Un JSON produit avant le 18/11 n'en a pas :
    # on fait alors comme si le texte entier tenait sur une seule ligne.
    lignes = zone.get("lignes", [])
    if len(lignes) == 0:
        lignes = [{"texte": zone["texte"]}]

    # Le numéro du jour est le premier mot de la 1re ligne, s'il ne contient que 1 ou 2 chiffres :
    # « 27 » oui ; « 9h », « 09:00 » ou « 0900 » non (ce sont des heures).
    premiere_ligne = lignes[0]
    mots = premiere_ligne["texte"].split()
    commence_par_un_numero = len(mots) > 0 and mots[0].isdigit() and len(mots[0]) <= 2
    if not commence_par_un_numero:
        return None, [zone]  # planning hebdo : la case entière est un seul événement

    numero = int(mots[0])
    evenements = []

    # Le reste de la 1re ligne : le 1er événement, si l'OCR l'a lu sur la même ligne que le numéro
    reste = " ".join(mots[1:])
    if reste != "":
        ligne_sans_numero = dict(premiere_ligne)  # une copie de la ligne (pour ne pas modifier le JSON)...
        ligne_sans_numero["texte"] = reste        # ...dont on retire le numéro du jour
        evenements.append(ligne_sans_numero)

    # Chacune des lignes suivantes est un événement
    for ligne in lignes[1:]:
        evenements.append(ligne)

    return numero, evenements  # liste vide : jour sans événement

In [ ]:
# Test de associer_ligne et associer_colonne : de petits plannings dessinés à la main, puis vos vraies images.
# Un planning classique : les noms à gauche (en tête de ligne), les dates en haut (en tête de colonne).
#
#            ┌ 23/03 ┐┌ 24/03 ┐      x : 110 à 190, puis 190 à 270 ; y : 0 à 30
#  Roux Inès  [ M06 ]  [ RH  ]       ligne de Roux Inès   : y 38 à 62
#  Le Gall    [ N08 ]  [ M06 ]       ligne de Le Gall Tom : y 64 à 88

roux = {"x1": 0, "y1": 38, "x2": 100, "y2": 62, "texte": "Roux Inès"}
le_gall = {"x1": 0, "y1": 64, "x2": 100, "y2": 88, "texte": "Le Gall Tom"}
date_23 = {"x1": 110, "y1": 0, "x2": 190, "y2": 30, "texte": "23/03"}
date_24 = {"x1": 190, "y1": 0, "x2": 270, "y2": 30, "texte": "24/03"}
m06 = {"x1": 120, "y1": 40, "x2": 180, "y2": 60, "texte": "M06"}  # Roux Inès, le 23/03
n08 = {"x1": 120, "y1": 66, "x2": 180, "y2": 86, "texte": "N08"}  # Le Gall Tom, le 23/03
rh = {"x1": 200, "y1": 40, "x2": 260, "y2": 60, "texte": "RH"}    # Roux Inès, le 24/03


def test_ligne():
    """associer_ligne trouve l'en-tête de la ligne (ici le nom), à gauche de l'événement."""
    assert associer_ligne(m06, [roux, le_gall])["texte"] == "Roux Inès"
    assert associer_ligne(n08, [roux, le_gall])["texte"] == "Le Gall Tom"
    assert associer_ligne(rh, [le_gall, roux])["texte"] == "Roux Inès"  # l'ordre de la liste ne compte pas
    assert associer_ligne(m06, []) is None  # aucune zone : pas de réponse
    entre_deux = {"x1": 120, "y1": 52, "x2": 180, "y2": 76, "texte": "?"}  # milieu y=64 : entre les deux lignes
    assert associer_ligne(entre_deux, [roux]) is None
    a_droite = {"x1": 300, "y1": 38, "x2": 400, "y2": 62, "texte": "à droite"}  # même ligne, mais pas à gauche
    assert associer_ligne(m06, [a_droite]) is None


def test_colonne():
    """associer_colonne trouve l'en-tête de la colonne (ici la date), au-dessus de l'événement."""
    assert associer_colonne(m06, [date_23, date_24])["texte"] == "23/03"
    assert associer_colonne(rh, [date_23, date_24])["texte"] == "24/03"
    assert associer_colonne(n08, [date_24, date_23])["texte"] == "23/03"  # l'ordre de la liste ne compte pas
    assert associer_colonne(m06, []) is None
    en_dessous = {"x1": 110, "y1": 100, "x2": 190, "y2": 130, "texte": "en dessous"}  # même colonne, mais pas au-dessus
    assert associer_colonne(m06, [en_dessous]) is None


def test_transpose():
    """Planning transposé : les mêmes fonctions, dans l'autre sens (nom en haut, date à gauche)."""
    nom_en_haut = {"x1": 110, "y1": 0, "x2": 190, "y2": 30, "texte": "M Lefèvre"}
    date_a_gauche = {"x1": 0, "y1": 38, "x2": 100, "y2": 62, "texte": "23 mars"}
    assert associer_colonne(m06, [nom_en_haut])["texte"] == "M Lefèvre"
    assert associer_ligne(m06, [date_a_gauche])["texte"] == "23 mars"


def test_decouper():
    """decouper_cellule : le numéro du jour, puis un événement par ligne (lignes entières, avec leur confiance)."""
    case_du_27 = {"texte": "27 9h15 Atelier", "lignes": [{"texte": "27", "confiance": 0.99},
                                                          {"texte": "9h15 Atelier", "confiance": 0.78}]}
    numero, evenements = decouper_cellule(case_du_27)
    assert numero == 27
    assert evenements[0]["texte"] == "9h15 Atelier"
    assert evenements[0]["confiance"] == 0.78  # la ligne entière, pas seulement son texte
    numero, evenements = decouper_cellule({"texte": "16 10:15 Coiffeur 18:00 Cinéma",
                                           "lignes": [{"texte": "16 10:15 Coiffeur"}, {"texte": "18:00 Cinéma"}]})
    assert numero == 16 and [e["texte"] for e in evenements] == ["10:15 Coiffeur", "18:00 Cinéma"]
    assert decouper_cellule({"texte": "23", "lignes": [{"texte": "23"}]}) == (23, [])  # jour vide
    hebdo = {"texte": "9h à 18h Hangar C", "lignes": [{"texte": "9h à 18h"}, {"texte": "Hangar C"}]}
    assert decouper_cellule(hebdo) == (None, [hebdo])  # planning hebdo : la case entière, telle quelle
    assert decouper_cellule({"texte": "", "lignes": []}) == (None, [])  # l'OCR n'a rien lu


for nom, f in [("associer_ligne", test_ligne), ("associer_colonne", test_colonne), ("planning transposé", test_transpose),
               ("decouper_cellule", test_decouper)]:
    try:
        f()
        print(f"OK      {nom}")
    except NameError:
        print(f"ABSENTE {nom} : la fonction n'est pas encore dans le notebook")
    except AssertionError:
        print(f"ECHEC   {nom}")

# Sur vos vraies images : les 3 premiers événements de chaque planning en tableau, avec leur personne et leur date.
# Pas de vérification automatique ici (les textes dépendent de l'OCR) : regardez si c'est cohérent avec la photo.
print()
for image in ocr["images"]:
    noms = [z for z in image["zones"] if z["classe"] == "Nom"]
    dates = [z for z in image["zones"] if z["classe"] == "Date"]
    evenements = [z for z in image["zones"] if z["classe"] in ("Cellule", "Cellule_code") and z["texte"]]
    if len(dates) == 0:
        print(image["image"], ": grille de mois, la date est dans la Cellule (pas de personne)")
        for case in evenements[:3]:
            numero, lignes_evt = decouper_cellule(case)
            textes = [ligne["texte"] for ligne in lignes_evt]
            print(f"    jour {numero} : {textes}")
        continue
    # Planning transposé si les dates sont les unes SOUS les autres (plus d'écart en hauteur qu'en largeur)
    ecart_haut = max(d["y1"] for d in dates) - min(d["y1"] for d in dates)
    ecart_large = max(d["x1"] for d in dates) - min(d["x1"] for d in dates)
    transpose = ecart_haut > ecart_large
    print(image["image"], ": planning transposé" if transpose else ": planning classique")
    for evt in evenements[:3]:
        try:
            if transpose:
                personne, date = associer_colonne(evt, noms), associer_ligne(evt, dates)
            else:
                personne, date = associer_ligne(evt, noms), associer_colonne(evt, dates)
        except NameError:
            print("    associer_colonne absente : ajoutez-la pour voir les résultats")
            break
        texte_personne = personne["texte"] if personne else None
        texte_date = date["texte"] if date else None
        print(f"    {evt['texte'][:28]!r:32} personne = {texte_personne!r:18} date = {texte_date!r}")

### 3.4 bis Plannings à codes : `traduire_code(texte)`

Dans un planning d'équipe, la cellule ne contient pas des heures mais un **code** (`M06`, `N08`, `RH`...).
La **table des codes** est fournie avec le planning (légende) : à vous de la recopier dans `CODES`.

| Catégorie | Exemple | Résultat |
|---|---|---|
| travail | `M06` (06:00-14:00) | créneau horaire |
| nuit | `N08` (20:00-08:00) | `overnight=True` : la fin tombe le lendemain |
| repos / congé | `RH` | `all_day=True`, sans heures |

> **RGPD** : un arrêt maladie (`MAL`) est une **donnée de santé**. Dans un agenda partagé, son titre doit
> être « **Absence** », jamais « Arrêt maladie ».

In [ ]:
# TODO : complétez avec TOUS les codes de la légende de VOTRE planning
CODES = {
    "M06": {"titre": "Matin", "debut": "06:00", "fin": "14:00"},
    "N08": {"titre": "Nuit", "debut": "20:00", "fin": "08:00"},   # fin < début : nuit
    "RH":  {"titre": "Repos hebdomadaire", "debut": None, "fin": None},  # repos : journée entière
    # TODO : MAL -> titre « Absence » (pas « Arrêt maladie »), journée entière
}


def traduire_code(texte: str):
    """Traduit un code de planning en (titre, start_time, end_time, all_day, overnight), ou None si inconnu.

    >>> traduire_code("M06")
    ('Matin', '06:00', '14:00', False, False)
    >>> traduire_code("MO6")                  # bruit d'OCR : le O est un 0
    ('Matin', '06:00', '14:00', False, False)
    >>> traduire_code("N08")
    ('Nuit', '20:00', '08:00', False, True)
    >>> traduire_code("RH")                   # repos : journée entière
    ('Repos hebdomadaire', None, None, True, False)
    >>> traduire_code("MAL")[0]               # donnée de santé : jamais dans l'agenda
    'Absence'
    >>> traduire_code("???") is None
    True
    """
    # TODO : nettoyez le texte (majuscules, espaces), corrigez le bruit (MO6 -> M06 : seulement les chiffres
    # attendus après la lettre), cherchez dans CODES. overnight = fin <= début ; all_day = pas d'heure.
    raise NotImplementedError

### 3.5 `confiance(...)` : combien fait-on confiance à l'événement ?

On part de **1,0** et on **retire** des points à chaque correction ou doute. À vous de choisir les pénalités
et de les **justifier** dans votre rapport. Idées :

- caractère corrigé par `normaliser` (O→0…) : petite pénalité ;
- confiance OCR faible de la zone (`confiance_ocr`) ou de la détection (`confiance_detection`) ;
- aucune heure trouvée (événement « toute la journée » par défaut) ;
- association à la date peu sûre (faible recouvrement) ;
- heure de fin ≤ début (erreur d'OCR).

Le résultat est borné entre 0 et 1.

In [ ]:
def confiance(zone_evt: dict, zone_date: dict | None, score_association: float,
              texte_brut: str, texte_normalise: str, heures: tuple) -> float:
    """Score de confiance de l'événement, entre 0 et 1.

    >>> 0.0 <= confiance({"confiance_ocr": 0.9, "confiance_detection": 0.9}, None, 0.0, "x", "x", (None, None)) <= 1.0
    True
    """
    score = 1.0
    # TODO : retirez des points selon les critères ci-dessus (une constante de pénalité par critère,
    # pour pouvoir les ajuster facilement), puis bornez : max(0.0, min(1.0, score)).
    raise NotImplementedError

### 3.6 `parser(ocr)` : assembler le tout

`parser` parcourt les images, associe chaque événement à sa date, extrait titre et heures, calcule la confiance,
et renvoie la liste des événements **au schéma du cours**. Le titre est le texte de la zone sans les heures.

In [ ]:
def parser(ocr: dict, annee: int = ANNEE) -> list[dict]:
    """Transforme le JSON de l'OCR en liste d'événements (schéma du cours).

    Chaque élément : {"title", "date", "start_time", "end_time", "location", "confidence"}
    (+ "person", "all_day", "overnight" pour un planning à codes).
    Un événement dont la date est introuvable n'est pas perdu : mettez date=None et une confiance basse
    (ou écartez-le de façon explicite : décrivez votre choix dans le rapport).
    """
    evenements = []
    for image in ocr["images"]:
        zones = image["zones"]
        # TODO :
        # 1. repérer le mois / l'année dans les en-têtes de mois (extraire_mois_annee) ;
        # 2. pour chaque (zone_evt, zone_date, score) de associer(zones) :
        #      - normaliser le texte, extraire_heures, extraire_date (avec le mois connu) ;
        #      - title = texte sans les heures, nettoyé ;
        #      - planning à codes : person = associer_ligne(...), (title, heures, all_day, overnight) = traduire_code(...) ;
        #      - confidence = confiance(...) ;
        # 3. ajouter le dict à `evenements`.
        pass
    return evenements

## 4. Tests : faites-les passer

Quelques cas écrits à la main, dont du **bruit d'OCR**. Quand une fonction n'est pas encore écrite,
le test correspondant échoue avec `NotImplementedError` : normal. Avancez fonction par fonction.

In [ ]:
def tester(nom, fonction):
    try:
        fonction()
        print(f"OK     {nom}")
    except NotImplementedError:
        print(f"A FAIRE {nom}")
    except AssertionError as e:
        print(f"ECHEC  {nom} : {e}")


def test_normaliser():
    assert normaliser("1Z/O3") == "12/03"
    assert normaliser("I4h-18h") == "14h-18h"
    assert normaliser("Octobre") == "Octobre"


def test_heures():
    assert extraire_heures("14h-18h") == ("14:00", "18:00")
    assert extraire_heures("14:00-18:00") == ("14:00", "18:00")
    assert extraire_heures("de 14h à 18h") == ("14:00", "18:00")
    assert extraire_heures("9h30") == ("09:30", None)
    assert extraire_heures("Congé") == (None, None)
    assert extraire_heures("I4h-18h") == ("14:00", "18:00")  # bruit d'OCR : pensez à normaliser d'abord


def test_dates():
    assert extraire_date("12/03/2025") == "2025-03-12"
    assert extraire_date("12", mois=3) == "2026-03-12"
    assert extraire_date("31/02") is None


def test_bruit_ocr():
    # Le cas de référence : date ET heures dans un texte bruité
    texte = "Mar . 1Z/O3  I4h-18h"
    assert extraire_date(texte) == "2026-03-12", extraire_date(texte)
    assert extraire_heures(texte) == ("14:00", "18:00"), extraire_heures(texte)


def test_association():
    evt = {"classe": "event", "x1": 12, "y1": 30, "x2": 90, "y2": 50, "texte": "14h-18h A"}
    j12 = {"classe": "date_cell", "x1": 0, "y1": 0, "x2": 100, "y2": 100, "texte": "12"}
    j13 = {"classe": "date_cell", "x1": 100, "y1": 0, "x2": 200, "y2": 100, "texte": "13"}
    assert associer([evt, j12, j13])[0][1]["texte"] == "12"


def test_confiance():
    z = {"confiance_ocr": 0.95, "confiance_detection": 0.9}
    propre = confiance(z, {}, 1.0, "14h-18h", "14h-18h", ("14:00", "18:00"))
    bruite = confiance(z, {}, 1.0, "I4h-18h", "14h-18h", ("14:00", "18:00"))
    assert 0 <= bruite < propre <= 1, (bruite, propre)  # une correction doit coûter de la confiance


def test_associer_ligne():
    evt = {"x1": 120, "y1": 40, "x2": 180, "y2": 60, "texte": "M06"}
    p1 = {"x1": 0, "y1": 38, "x2": 100, "y2": 62, "texte": "Roux Inès"}
    p2 = {"x1": 0, "y1": 64, "x2": 100, "y2": 88, "texte": "Le Gall Tom"}
    assert associer_ligne(evt, [p2, p1])["texte"] == "Roux Inès"
    assert associer_ligne(evt, []) is None


def test_codes():
    assert traduire_code("M06") == ("Matin", "06:00", "14:00", False, False)
    assert traduire_code("MO6") == ("Matin", "06:00", "14:00", False, False)  # bruit d'OCR
    titre, debut, fin, journee, nuit = traduire_code("N08")
    assert nuit and not journee and (debut, fin) == ("20:00", "08:00")
    assert traduire_code("RH")[3] is True  # repos : journée entière
    assert traduire_code("MAL")[0] == "Absence"  # RGPD : pas de donnée de santé
    assert traduire_code("???") is None


for nom, f in [("normaliser", test_normaliser), ("extraire_heures", test_heures),
               ("extraire_date", test_dates), ("bruit d'OCR", test_bruit_ocr),
               ("associer", test_association), ("confiance", test_confiance),
               ("associer_ligne", test_associer_ligne), ("traduire_code", test_codes)]:
    tester(nom, f)

## 5. Lancer le parsing sur vos résultats OCR

In [ ]:
evenements = parser(ocr)
print(len(evenements), "événement(s) extrait(s)")
if not evenements:
    print("ATTENTION : 0 événement. parser() n'est pas encore écrit (section 3.6), "
          "ou il ne reconnaît pas vos classes (section 3, CLASSES_...).")
for e in evenements[:5]:
    print(e)

## 6. Tableau de revue : trois bandes de confiance

- **≥ 0,8** : pré-coché ;
- **0,5 – 0,8** : à vérifier (relisez-les : le parsing a corrigé ou deviné quelque chose) ;
- **< 0,5** : non proposé.

Vous ne publiez que les événements pré-cochés **et** ceux que vous avez vérifiés.

In [ ]:
SEUIL_PRECOCHE = 0.8
SEUIL_PROPOSE = 0.5

precoches = [e for e in evenements if e["confidence"] >= SEUIL_PRECOCHE]
a_verifier = [e for e in evenements if SEUIL_PROPOSE <= e["confidence"] < SEUIL_PRECOCHE]
non_proposes = [e for e in evenements if e["confidence"] < SEUIL_PROPOSE]

try:
    import pandas as pd
    for titre, liste in [("Pré-cochés", precoches), ("À vérifier", a_verifier), ("Non proposés", non_proposes)]:
        print(f"\n== {titre} : {len(liste)}")
        if liste:
            display(pd.DataFrame(liste).sort_values(["date", "start_time"], na_position="last"))
except ImportError:  # pandas absent : affichage simple
    for titre, liste in [("Pré-cochés", precoches), ("À vérifier", a_verifier), ("Non proposés", non_proposes)]:
        print(f"\n== {titre} : {len(liste)}")
        for e in liste:
            print(" ", e)

## 7. Publication dans Radicale (fournie)

Le module `caldav_publier.py` envoie vos événements dans l'agenda **`calevent`** de votre squad
(l'agenda partagé ; votre agenda personnel n'est pas touché). Il garantit :

- des heures en `Europe/Paris` et un identifiant (UID) stable : renvoyer le même créneau **ne crée pas de doublon** ;
- un événement « toute la journée » quand il n'y a pas d'heure de début, une durée de 1 h sans heure de fin.

**Où trouver le module ?**

- en local : lancez ce notebook depuis le dossier `ressources/` (ou copiez `caldav_publier.py` à côté du notebook) ;
- dans Colab : déposez le fichier dans le panneau *Fichiers*, ou téléchargez-le avec la cellule ci-dessous.

In [ ]:
%pip install -q caldav icalendar

# Colab uniquement, si caldav_publier.py n'est pas à côté du notebook :
if not Path("caldav_publier.py").exists():
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/CassiaFa/flexistaff/main/ressources/caldav_publier.py",
        "caldav_publier.py")

### Simulation d'abord (aucune connexion)

`dry_run=True` construit les événements iCalendar et affiche le statut `simulation` : un bon moyen de repérer
une date absente ou une heure invalide (statut `erreur: ...`) avant d'écrire dans l'agenda de l'équipe.

In [ ]:
from caldav_publier import publier, charger_identifiants

a_publier = precoches + a_verifier  # TODO : ne gardez dans a_verifier que ce que vous avez relu
simulation = publier(a_publier, url="", utilisateur="", mot_de_passe="", dry_run=True)
for r in simulation:
    print(f"{r['statut']:<12} {r['date']}  {r['titre']}")

### Publication réelle

Passez `PUBLIER = True` dans les paramètres **seulement** quand la simulation est propre.
Les statuts possibles : `créé`, `déjà présent` (relancer la cellule est sans danger), `erreur: ...`.
Le fichier d'identifiants n'est jamais affiché ni écrit dans le code.

In [ ]:
if PUBLIER:
    ident = charger_identifiants(CREDENTIALS)          # url, utilisateur, mot_de_passe
    resultats = publier(a_publier, **ident)             # agenda "calevent" par défaut
    for r in resultats:
        print(f"{r['statut']:<14} {r['date']}  {r['titre']}")
    with open(CREDENTIALS, encoding="utf-8") as f:
        print("\nVérifiez le résultat dans l'interface web de Radicale :", json.load(f)["caldav"]["web_url"])
else:
    print("PUBLIER = False : rien n'a été envoyé. Passez-le à True après la simulation.")

## 8. Questions pour votre rapport du jour 3

Répondez dans vos livrables, **chiffres à l'appui** (sur les images du jeu de test) :

1. **Seuils** : quels seuils de confiance avez-vous retenus (0,8 / 0,5 ou d'autres) ? Pourquoi ?
   Combien d'événements dans chaque bande ? En avez-vous vu un mal classé (bon événement non proposé, ou faux événement pré-coché) ?
2. **Pénalités** : quelles pénalités avez-vous choisies dans `confiance` et pourquoi ces valeurs ?
3. **Rappel** : combien de créneaux réels manquent à la fin (zone non détectée, texte vide, date introuvable) ?
   À quelle étape se perdent-ils : détection, OCR ou parsing ?
4. **Ce qui échoue** : quels textes OCR votre parser ne sait pas lire ? Donnez 3 exemples concrets et une piste d'amélioration.
5. **Association** : quelle règle géométrique avez-vous utilisée pour relier événement et date ? Dans quel cas se trompe-t-elle ?
6. **Détection et parsing** : le challenger aurait-il changé le résultat ? Qu'auriez-vous demandé au modèle de mieux détecter ?
7. **Bonus (optionnel)** : exposez `parser` dans une petite API FastAPI `POST /parse` (voir le guide).

## Annexe : pas à pas sur une image

Pour comprendre ce que reçoit `parser`, on suit **une seule image**, une petite étape par cellule. La dernière ligne de chaque cellule s'affiche toute seule : lancez-les dans l'ordre et regardez le résultat.

À lancer après les sections 1 et 2 (qui chargent `ocr`). Pour une autre image, changez `IMAGE` dans la cellule A et relancez A, B, C, D.

In [34]:
# A. Choisir l'image
IMAGE = 7  # p007, le planning à codes ; changez le numéro pour un autre type
image = ocr["images"][IMAGE]
image["image"]

'lot-test_p007_IMG_2916.jpg'

In [35]:
# B. Une liste par classe : on garde les zones dont la classe correspond
cellules = [zone for zone in image["zones"] if zone["classe"] == "Cellule"]
codes = [zone for zone in image["zones"] if zone["classe"] == "Cellule_code"]
dates = [zone for zone in image["zones"] if zone["classe"] == "Date"]
noms = [zone for zone in image["zones"] if zone["classe"] == "Nom"]
titres = [zone for zone in image["zones"] if zone["classe"] == "Titre"]
len(cellules), len(codes), len(dates), len(noms), len(titres)

(0, 166, 15, 12, 0)

In [39]:
# C. Regarder une liste de près : ici, le texte lu dans chaque date
[date["texte"] for date in dates]
[code["texte"] for code in codes]

['G03',
 'RC',
 'J09',
 '514',
 'CA',
 'G03',
 'RC',
 'J09',
 'NoB',
 'J09',
 'J09',
 'J09',
 'CA',
 '',
 '',
 'G03',
 '',
 'MO6',
 'G03',
 'J09',
 'Mo6',
 'CA',
 'RC',
 'C03',
 'RH',
 'NOB',
 'RF',
 'S14',
 'CA',
 'G03',
 'FOR',
 'RC',
 'C03',
 'RC',
 '',
 'RF',
 'RH',
 '',
 'NO8',
 'RH',
 'J09',
 'MOG',
 '',
 'RH',
 'RH',
 'CA',
 'RH',
 'NOB',
 'G03',
 'G05',
 'G03',
 'RC',
 '',
 'RH',
 'RH',
 '514',
 'CA',
 'RC',
 'RH',
 'CA',
 'RH',
 'CA',
 'MO6',
 '514',
 'MO6',
 '',
 'NOB',
 'NOB',
 'G03',
 'CA',
 '',
 'S14',
 'S14',
 'CA',
 '',
 'RH',
 'RF',
 '',
 'RH',
 'RH',
 'CA',
 'G03',
 'G03',
 'RC',
 'RC',
 'RF',
 'CA',
 'Go5',
 'CA',
 '',
 'RH',
 'RC',
 '514',
 'FOR',
 'RH',
 '',
 'J09',
 'GA',
 '',
 'RC',
 '514',
 'RC',
 'RH',
 'GO5',
 'RH',
 'RH',
 'NoB',
 '514',
 'RH',
 'RH',
 '',
 '',
 'CA',
 'RC',
 'GO5',
 'RH',
 'CA',
 'RH',
 'CA',
 '',
 'MO6',
 'MO6',
 'S14',
 'NOB',
 'Mo6',
 'RC',
 'G03',
 'RH',
 'CA',
 '',
 'NO8',
 'J09',
 'G03',
 'NOB',
 'G03',
 'FOR',
 'G03',
 'RH',
 'RC',
 'R

In [37]:
# D. Le type de planning, d'après les classes présentes
if len(codes) > 0:
    type_planning = "planning à codes"
elif len(dates) == 0:
    type_planning = "grille de mois"  # pas de Date : la date est dans la Cellule
else:
    # Les dates les unes SOUS les autres (plus d'écart en hauteur qu'en largeur) : planning transposé
    ecart_hauteur = max(d["y1"] for d in dates) - min(d["y1"] for d in dates)
    ecart_largeur = max(d["x1"] for d in dates) - min(d["x1"] for d in dates)
    if ecart_hauteur > ecart_largeur:
        type_planning = "planning hebdo transposé"
    else:
        type_planning = "planning hebdo"
type_planning

'planning à codes'